# EMD-Based Depression Classification — Generic Pipeline

This notebook implements a **sex-independent, fully configurable** classification pipeline for speech-based depression detection using Empirical Mode Decomposition (EMD).

**Pipeline overview:**
1. Load pre-computed IMF arrays (depression vs. healthy group)
2. Select discriminative IMFs using the **Gaussian Kernel (RBF) criterion**
3. Extract statistical features (skewness, kurtosis, median, std, mean) per IMF window
4. Balance classes and split into train/test sets
5. Train and evaluate **six classifiers** (RF, GB, SVM, LR, KNN, GNB) via GridSearchCV
6. Visualise confusion matrices, ROC curves, and a comparative results table

All paths, hyperparameters, and group labels are set in the **Configuration** section.

## 1. Imports

Load all required libraries for data handling, statistical analysis, machine learning, and visualisation.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, roc_curve, auc
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

print("Libraries loaded successfully.")

## 2. Configuration

All user-defined parameters are centralised here. Adjust the paths to the `.npy` data files, the group label (e.g. `'Female'` or `'Male'`), and the modelling hyperparameters before running the notebook.

| Parameter | Description |
|---|---|
| `DEP_DATA_PATH` | Path to the depression-group IMF array |
| `HEALTH_DATA_PATH` | Path to the healthy-group IMF array |
| `GROUP_LABEL` | Label used in plot titles (e.g. `'Female'`, `'Male'`) |
| `N_IMFS` | Total number of IMFs in the arrays |
| `GK_SIGMA` | Bandwidth σ for the Gaussian kernel |
| `GK_THRESHOLD` | Similarity threshold; IMFs below this are retained |
| `WINDOW_SIZE` | Number of samples per sub-window for feature extraction |
| `TEST_SIZE` | Fraction of balanced data held out for testing |
| `RANDOM_STATE` | Global random seed for reproducibility |
| `CV_FOLDS` | Number of folds in the inner GridSearchCV cross-validation |

In [ ]:
# ── DATA PATHS ──────────────────────────────────────────────────────────────
DEP_DATA_PATH    = "path/to/depression_imfs.npy"   # shape: (n_subjects, N_IMFS, n_samples)
HEALTH_DATA_PATH = "path/to/healthy_imfs.npy"       # shape: (n_subjects, N_IMFS, n_samples)

# ── GROUP LABEL (used in plots and prints) ───────────────────────────────────
GROUP_LABEL = "Group"   # e.g. 'Female', 'Male', 'All'

# ── IMF SETTINGS ─────────────────────────────────────────────────────────────
N_IMFS = 16             # total number of IMFs per subject

# ── GAUSSIAN KERNEL (GK) SELECTION ───────────────────────────────────────────
GK_SIGMA     = 0.5      # bandwidth parameter σ
GK_THRESHOLD = 1e-40    # IMFs with mean similarity ≤ threshold are selected

# ── FEATURE EXTRACTION ───────────────────────────────────────────────────────
WINDOW_SIZE = 10_000    # samples per sub-window

# ── TRAIN / TEST SPLIT ───────────────────────────────────────────────────────
TEST_SIZE    = 0.2
RANDOM_STATE = 42

# ── CROSS-VALIDATION ─────────────────────────────────────────────────────────
CV_FOLDS = 5

print(f"Configuration set for group: '{GROUP_LABEL}'")
print(f"  Depression data : {DEP_DATA_PATH}")
print(f"  Healthy data    : {HEALTH_DATA_PATH}")
print(f"  N_IMFS={N_IMFS} | GK σ={GK_SIGMA} | GK threshold={GK_THRESHOLD}")
print(f"  Window size={WINDOW_SIZE} | Test size={TEST_SIZE} | Seed={RANDOM_STATE}")

## 3. Data Loading

Load the pre-computed IMF arrays for the depression and healthy groups from disk.

Expected array shape: **(n_subjects, N_IMFS, n_samples)**
- `n_subjects`: number of participants in each group
- `N_IMFS`: number of EMD-derived intrinsic mode functions (default 16)
- `n_samples`: total number of signal samples per subject

In [ ]:
dep_imfs    = np.load(DEP_DATA_PATH,    allow_pickle=True)
health_imfs = np.load(HEALTH_DATA_PATH, allow_pickle=True)

print(f"Depression group shape : {dep_imfs.shape}   (subjects × IMFs × samples)")
print(f"Healthy group shape    : {health_imfs.shape}")

## 4. Convert to DataFrame

Reshape the 3-D arrays into 2-D DataFrames where each row is a subject and each column contains the full signal for one IMF. This format is required by the Gaussian kernel comparison function in the next step.

In [ ]:
imf_columns = [f"imf{i+1}" for i in range(N_IMFS)]

df_dep    = pd.DataFrame.from_records(dep_imfs,    columns=imf_columns)
df_health = pd.DataFrame.from_records(health_imfs, columns=imf_columns)

print(f"Depression DataFrame  : {df_dep.shape}")
print(f"Healthy DataFrame     : {df_health.shape}")

## 5. Gaussian Kernel (GK) Discriminability Criterion

The **Radial Basis Function (RBF) Gaussian kernel** quantifies the cross-class distributional similarity for each IMF:

$$K(x, x') = \exp\!\left(-\frac{\|x - x'\|^2}{2\sigma^2}\right)$$

A **lower** mean kernel value indicates **greater distributional discrepancy** between the depression and healthy groups for that IMF. IMFs whose mean similarity falls at or below `GK_THRESHOLD` are retained as discriminative features.

This is the same criterion described in Section 2.4.3 of the thesis.

In [ ]:
def gaussian_kernel(x, y, sigma):
    """Point-wise RBF Gaussian kernel between two 1-D arrays."""
    min_len = min(len(x), len(y))
    x, y = x[:min_len], y[:min_len]
    return np.exp(-np.sum((x - y) ** 2) / (2 * sigma ** 2))


def select_imfs_by_gk(df_dep, df_health, sigma=GK_SIGMA, threshold=GK_THRESHOLD):
    """
    Compute the mean cross-class Gaussian kernel similarity for each IMF
    and return the 0-based indices of IMFs below the discriminability threshold.

    Parameters
    ----------
    df_dep, df_health : pd.DataFrame
        Subject-level DataFrames (one column per IMF, values are 1-D arrays).
    sigma : float
        Bandwidth parameter for the RBF kernel.
    threshold : float
        Similarity threshold; IMFs with mean similarity ≤ threshold are selected.

    Returns
    -------
    similarities : list of float
    selected_idx : list of int (0-based column indices)
    """
    similarities = []
    selected_idx = []
    n_imfs = df_dep.shape[1]

    for i in range(n_imfs):
        sims = [
            gaussian_kernel(df_dep.iloc[r, i], df_health.iloc[s, i], sigma)
            for r in range(len(df_dep))
            for s in range(len(df_health))
        ]
        avg_sim = np.mean(sims)
        similarities.append(avg_sim)
        if avg_sim <= threshold:
            selected_idx.append(i)

    return similarities, selected_idx


print(f"Running Gaussian Kernel selection (σ={GK_SIGMA}, threshold={GK_THRESHOLD})...")
gk_similarities, selected_imf_idx = select_imfs_by_gk(df_dep, df_health)

selected_imf_names = [imf_columns[i] for i in selected_imf_idx]
print(f"\nGK similarities per IMF:")
for name, sim in zip(imf_columns, gk_similarities):
    marker = " ← selected" if sim <= GK_THRESHOLD else ""
    print(f"  {name}: {sim:.4e}{marker}")

print(f"\nSelected IMFs (0-based indices) : {selected_imf_idx}")
print(f"Selected IMF names              : {selected_imf_names}")

### 5.1 Visualise GK Scores

Bar chart of mean cross-class Gaussian kernel similarity per IMF. Selected IMFs (below threshold) are highlighted in green; discarded ones in grey. The dashed red line marks the selection threshold.

In [ ]:
colors = ['#4CAF50' if i in selected_imf_idx else '#9E9E9E' for i in range(N_IMFS)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(imf_columns, gk_similarities, color=colors, edgecolor='white')
ax.axhline(GK_THRESHOLD, color='red', linestyle='--', linewidth=1.2,
           label=f'Selection threshold ({GK_THRESHOLD:.0e})')
ax.set_xlabel('IMF', fontsize=12)
ax.set_ylabel('Mean RBF similarity', fontsize=12)
ax.set_title(f'Gaussian Kernel Cross-Class Similarity — {GROUP_LABEL}', fontsize=13)
ax.set_yscale('log')
ax.tick_params(axis='x', rotation=45)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#4CAF50', label='Selected'),
    Patch(facecolor='#9E9E9E', label='Discarded'),
]
ax.legend(handles=legend_elements + ax.get_legend_handles_labels()[0][-1:], fontsize=10)
plt.tight_layout()
plt.show()

## 6. Feature Extraction

For each subject and each **selected IMF**, the full signal is divided into non-overlapping windows of `WINDOW_SIZE` samples. Five statistical descriptors are computed per window:

| Descriptor | Function |
|---|---|
| Skewness | `scipy.stats.skew` |
| Kurtosis | `scipy.stats.kurtosis` |
| Median | `np.median` |
| Standard deviation | `np.std` |
| Mean | `np.mean` |

The window-level statistics are **averaged** across windows to produce a single feature vector per subject. Final feature dimensionality: `n_selected_IMFs × 5`.

In [ ]:
def extract_statistics(imfs_3d, selected_idx, window_size=WINDOW_SIZE):
    """
    Extract windowed statistical features from a 3-D IMF array.

    Parameters
    ----------
    imfs_3d : np.ndarray, shape (n_subjects, N_IMFS, n_samples)
    selected_idx : list of int
        0-based column indices of the IMFs to use.
    window_size : int
        Number of samples per sub-window.

    Returns
    -------
    features : np.ndarray, shape (n_subjects, len(selected_idx) * 5)
    """
    n_subjects = imfs_3d.shape[0]
    n_sel      = len(selected_idx)
    n_samples  = imfs_3d.shape[2]
    n_windows  = n_samples // window_size

    if n_windows == 0:
        raise ValueError(
            f"window_size={window_size} is larger than n_samples={n_samples}. "
            "Reduce WINDOW_SIZE."
        )

    features = np.zeros((n_subjects, n_sel * 5))

    for i in range(n_subjects):
        col = 0
        for j in selected_idx:
            signal = imfs_3d[i, j]
            win_stats = []
            for w in range(n_windows):
                seg = signal[w * window_size:(w + 1) * window_size]
                win_stats.append([
                    skew(seg),
                    kurtosis(seg),
                    np.median(seg),
                    np.std(seg),
                    np.mean(seg),
                ])
            win_stats = np.array(win_stats)           # (n_windows, 5)
            features[i, col:col + 5] = win_stats.mean(axis=0)
            col += 5

    return features


print("Extracting features for depression group...")
X_dep    = extract_statistics(dep_imfs,    selected_imf_idx)
print("Extracting features for healthy group...")
X_health = extract_statistics(health_imfs, selected_imf_idx)

print(f"\nFeature matrix — depression : {X_dep.shape}")
print(f"Feature matrix — healthy    : {X_health.shape}")
print(f"Features per subject: {len(selected_imf_idx)} IMFs × 5 stats = {len(selected_imf_idx)*5}")

## 7. Data Preparation — Balancing and Train/Test Split

**Class balancing:** Random undersampling of the majority class to produce equal class sizes. This avoids classifier bias towards the dominant class and ensures that accuracy is a meaningful metric.

**Labels:** Depression = 1, Healthy = 0

**Split:** `TEST_SIZE` fraction of the balanced dataset is held out as the test set (same random seed for reproducibility).

In [ ]:
np.random.seed(RANDOM_STATE)

# ── Concatenate and label ────────────────────────────────────────────────────
X_all = np.vstack([X_dep, X_health])
y_all = np.concatenate([
    np.ones(len(X_dep),    dtype=int),   # Depression = 1
    np.zeros(len(X_health), dtype=int),  # Healthy    = 0
])

print(f"Full dataset: {X_all.shape[0]} subjects  "
      f"(Depression={len(X_dep)}, Healthy={len(X_health)})")


# ── Random undersampling ─────────────────────────────────────────────────────
def balance_classes(X, y, random_state=RANDOM_STATE):
    """Undersample the majority class to match the minority class size."""
    rng = np.random.default_rng(random_state)
    idx_0 = np.where(y == 0)[0]
    idx_1 = np.where(y == 1)[0]
    min_size = min(len(idx_0), len(idx_1))
    idx_0_s  = rng.choice(idx_0, min_size, replace=False)
    idx_1_s  = rng.choice(idx_1, min_size, replace=False)
    idx_bal  = np.concatenate([idx_0_s, idx_1_s])
    return X[idx_bal], y[idx_bal]


X_bal, y_bal = balance_classes(X_all, y_all)
print(f"Balanced dataset: {X_bal.shape[0]} subjects  "
      f"(Depression={y_bal.sum()}, Healthy={(y_bal==0).sum()})")


# ── Train / test split ───────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_bal,
)

print(f"\nTrain set : {X_train.shape[0]} subjects")
print(f"Test set  : {X_test.shape[0]} subjects")

## 8. Model Definition

Six classifiers are evaluated, each wrapped in a `scikit-learn Pipeline` that applies `StandardScaler` before fitting:

| Classifier | Abbreviation | Key hyperparameters searched |
|---|---|---|
| Random Forest | RF | `n_estimators`, `max_depth`, `min_samples_split` |
| Gradient Boosting | GB | `n_estimators`, `learning_rate`, `max_depth` |
| Support Vector Machine | SVM | `C`, `kernel`, `gamma` |
| Logistic Regression | LR | `C`, `penalty` |
| K-Nearest Neighbours | KNN | `n_neighbors`, `weights` |
| Gaussian Naive Bayes | GNB | `var_smoothing` |

Hyperparameter selection uses **5-fold cross-validation** on the training set, optimising for **F1 score**.

In [ ]:
MODELS = {
    'Random Forest': (
        RandomForestClassifier(random_state=RANDOM_STATE),
        {
            'n_estimators':    [50, 100, 200],
            'max_depth':       [None, 5, 10],
            'min_samples_split': [2, 5],
        }
    ),
    'Gradient Boosting': (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {
            'n_estimators':  [50, 100, 200],
            'learning_rate': [0.05, 0.1, 0.2],
            'max_depth':     [2, 3, 5],
        }
    ),
    'SVM': (
        SVC(probability=True, random_state=RANDOM_STATE),
        {
            'C':      [0.1, 1, 10, 100],
            'kernel': ['rbf', 'linear'],
            'gamma':  ['scale', 'auto'],
        }
    ),
    'Logistic Regression': (
        LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, solver='liblinear'),
        {
            'C':       [0.001, 0.01, 0.1, 1, 10, 100],
            'penalty': ['l1', 'l2'],
        }
    ),
    'KNN': (
        KNeighborsClassifier(),
        {
            'n_neighbors': [3, 5, 7, 9, 11, 15],
            'weights':     ['uniform', 'distance'],
            'metric':      ['euclidean', 'manhattan'],
        }
    ),
    'Gaussian Naive Bayes': (
        GaussianNB(),
        {
            'var_smoothing': np.logspace(0, -9, 10),
        }
    ),
}

print(f"Defined {len(MODELS)} classifiers: {', '.join(MODELS.keys())}")

## 9. Training — GridSearchCV

Each classifier is tuned with `GridSearchCV` using stratified `KFold` (inner loop on the training set only). The best-performing hyperparameter combination per model is then evaluated on the **held-out test set**.

Metrics reported:
- **Accuracy** — fraction of correct predictions
- **Sensitivity (Recall)** — true positive rate (depression correctly identified)
- **Specificity** — true negative rate (healthy correctly identified)
- **Precision** — positive predictive value
- **F1 Score** — harmonic mean of precision and recall
- **ROC-AUC** — area under the receiver operating characteristic curve

In [ ]:
cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

results = []

for model_name, (estimator, param_grid) in MODELS.items():
    print(f"\n{'─'*55}")
    print(f"  Training: {model_name}")
    print(f"{'─'*55}")

    # Prefix params with 'model__' for Pipeline compatibility
    param_grid_pipe = {f'model__{k}': v for k, v in param_grid.items()}

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  estimator),
    ])

    grid = GridSearchCV(
        pipeline, param_grid_pipe,
        cv=cv, scoring='f1',
        n_jobs=-1, verbose=0,
    )
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    y_pred     = best_model.predict(X_test)

    # ROC-AUC
    if hasattr(best_model, 'predict_proba'):
        y_score = best_model.predict_proba(X_test)[:, 1]
    else:
        y_score = best_model.decision_function(X_test)
    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_auc     = auc(fpr, tpr)

    # Specificity from confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    TN, FP = cm[0, 0], cm[0, 1]
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0

    best_params_clean = {k.replace('model__', ''): v
                         for k, v in grid.best_params_.items()}

    results.append({
        'Model':        model_name,
        '_estimator':   best_model,
        '_fpr':         fpr,
        '_tpr':         tpr,
        '_cm':          cm,
        '_y_score':     y_score,
        'Best params':  best_params_clean,
        'Accuracy':     accuracy_score(y_test, y_pred),
        'Sensitivity':  recall_score(y_test, y_pred),
        'Specificity':  specificity,
        'Precision':    precision_score(y_test, y_pred, zero_division=0),
        'F1 Score':     f1_score(y_test, y_pred),
        'ROC-AUC':      roc_auc,
    })

    print(f"  Best params  : {best_params_clean}")
    print(f"  Accuracy     : {results[-1]['Accuracy']:.3f}")
    print(f"  Sensitivity  : {results[-1]['Sensitivity']:.3f}")
    print(f"  Specificity  : {results[-1]['Specificity']:.3f}")
    print(f"  F1 Score     : {results[-1]['F1 Score']:.3f}")
    print(f"  ROC-AUC      : {results[-1]['ROC-AUC']:.3f}")

print(f"\n{'═'*55}")
print("  All models trained.")
print(f"{'═'*55}")

## 10. Confusion Matrices

Each panel shows the confusion matrix for one classifier on the **test set**. Rows = true labels; columns = predicted labels.

In [ ]:
n_models = len(results)
ncols    = 3
nrows    = (n_models + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows))
axes = axes.flatten()

class_names = ['Healthy', 'Depression']

for ax, res in zip(axes, results):
    sns.heatmap(
        res['_cm'], annot=True, fmt='d', cmap='Blues',
        xticklabels=class_names, yticklabels=class_names,
        ax=ax, cbar=False,
    )
    ax.set_title(f"{res['Model']}\nACC={res['Accuracy']:.2f}  F1={res['F1 Score']:.2f}",
                 fontsize=11)
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('True', fontsize=10)

# Hide empty subplots
for ax in axes[n_models:]:
    ax.set_visible(False)

fig.suptitle(f'Confusion Matrices — {GROUP_LABEL} (balanced, selected IMFs)',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 11. ROC Curves

All six classifiers are overlaid on a single ROC plot. The AUC value in the legend summarises discrimination capacity across all possible classification thresholds. The dashed diagonal represents a random classifier (AUC = 0.50).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for res in results:
    ax.plot(
        res['_fpr'], res['_tpr'],
        label=f"{res['Model']} (AUC = {res['ROC-AUC']:.2f})",
        linewidth=2,
    )

ax.plot([0, 1], [0, 1], linestyle='--', color='grey',
        linewidth=1.2, label='Random (AUC = 0.50)')

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title(
    f'ROC Curves — {GROUP_LABEL}, selected IMFs (balanced)',
    fontsize=13
)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## 12. Results Summary

Comparative table of all six classifiers on the test set, sorted by ROC-AUC (descending). The best-performing model per metric is highlighted.

Column definitions:
- **Sensitivity** = Recall = TP / (TP + FN)  
- **Specificity** = TN / (TN + FP)  
- **F1 Score** = 2 × (Precision × Recall) / (Precision + Recall)

In [ ]:
metric_cols = ['Accuracy', 'Sensitivity', 'Specificity', 'Precision', 'F1 Score', 'ROC-AUC']

results_df = (
    pd.DataFrame(results)[['Model'] + metric_cols]
    .sort_values('ROC-AUC', ascending=False)
    .reset_index(drop=True)
)

print(f"\n{'='*70}")
print(f"  RESULTS SUMMARY — {GROUP_LABEL} | Selected IMFs: {selected_imf_names}")
print(f"{'='*70}")
print(results_df.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print(f"{'='*70}\n")

# Highlight best per metric
styled = (
    results_df.style
    .highlight_max(subset=metric_cols, color='#c8e6c9')
    .format({col: '{:.3f}' for col in metric_cols})
    .set_caption(f'Best value per metric highlighted — {GROUP_LABEL}')
)
styled

## 13. Best Hyperparameters per Model

The table below shows the hyperparameter combination selected by GridSearchCV for each classifier. These were chosen to maximise cross-validated F1 score on the training set.

In [ ]:
params_df = pd.DataFrame([
    {'Model': r['Model'], **r['Best params']}
    for r in results
])

print("Best hyperparameters per model (selected by GridSearchCV on training set):")
display(params_df.fillna('—').set_index('Model'))